[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/GEM2_gecko/gem2.ipynb)

# GEM2: enzyme-constrained models in geckopy

A Python / **geckopy** port of the GECKO 3 MATLAB exercise (`gem2_stage*.mlx`).

The MATLAB exercise builds an enzyme-constrained model (**ecModel**) of the
yeast *Rhodotorula toruloides* with GECKO 3, and then tunes, constrains and
analyses it. The reconstruction stages (UniProt, BRENDA, DLKcat, kcat
assignment) need online databases and Docker, so here we **start from the
pre-built ecModel** and focus on the analysis: model tuning (Stage 3),
proteomics integration (Stage 4) and simulation (Stage 5). geckopy *can* do the
full reconstruction too (`make_ec_model`, `fuzzy_kcat_matching`, `run_dlkcat`,
…) — see the `tutorials/full_ecModel/protocol.py` in the geckopy repository.

**Learning goals**

- Load a GECKO ecModel into Python with geckopy and understand its structure.
- Understand how enzyme constraints and the protein pool are represented.
- Tune kcat values and the protein pool, integrate proteomics data.
- Simulate: maximum growth, minimal protein usage, ecModel vs conventional GEM.

> ### ⚠️ Key difference: direction of the protein reactions
> geckopy and the GECKO MATLAB toolbox represent the protein machinery with
> **opposite reaction directions**:
> | | GECKO MATLAB | geckopy |
> |---|---|---|
> | `usage_prot_<id>` | `prot_<id> → prot_pool`, bounds `(-1000, 0)` | `prot_pool → prot_<id>`, bounds `(0, 1000)` |
> | `prot_pool_exchange` | `prot_pool → ∅`, bounds `(-1000, 0)` | `∅ → prot_pool`, bounds `(0, 1000)` |
> | flux sign | **negative** | **positive** |
> | proteomics constraint | set `lb = -conc` | set `ub = conc` |
>
> So in geckopy you **minimize the (positive) `prot_pool_exchange`** to find
> minimal protein usage, and protein/usage fluxes are positive. geckopy's YAML
> reader normalises a MATLAB-written ecModel to the forward direction on load.

## Setup

geckopy is not on PyPI yet, so it is installed from GitHub. It depends on
`raven-python`; geckopy's `main` currently needs `raven-python` from its `main`
branch (the pinned `develop` branch lags behind), so we install that explicitly.

In [ ]:
import sys
!{sys.executable} -m pip install -q "git+https://github.com/SysBioChalmers/geckopy.git@main"
!{sys.executable} -m pip install -q --force-reinstall --no-deps "git+https://github.com/SysBioChalmers/raven-python.git@main"
!{sys.executable} -m pip install -q matplotlib pandas numpy

In [ ]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geckopy
from geckopy import (
    ModelAdapter, ModelParameters,
    load_ec_model, load_conventional_gem, set_prot_pool_size,
    sensitivity_tuning, enzyme_usage, report_enzyme_usage, map_rxns_to_conv,
    load_prot_data, load_flux_data, fill_enz_concs, constrain_enz_concs,
    calculate_f_factor, apply_flux_data_constraints, flexibilize_enz_concs,
)
print('geckopy loaded; cobra', __import__('cobra').__version__)

Download the model and data files into a GECKO-style project folder
(`ecRhto/models`, `ecRhto/data`).

In [ ]:
os.makedirs('ecRhto/models', exist_ok=True)
os.makedirs('ecRhto/data', exist_ok=True)
base = "https://raw.githubusercontent.com/SysBioChalmers/MESBcourse/main/exercises/GEM2_gecko"
for sub, fname in [('models','ecRhto.yml'), ('models','rhto.xml'),
                   ('data','abs_proteomics.txt'), ('data','fluxData.tsv')]:
    dst = f'ecRhto/{sub}/{fname}'
    if not os.path.exists(dst):
        !wget -q {base}/ecRhto/{sub}/{fname} -O {dst}
print('downloaded:', os.listdir('ecRhto/models'), os.listdir('ecRhto/data'))

### A note on the ecModel YAML

The pre-built `ecRhto.yml` was written by GECKO **MATLAB**, whose YAML writer
emits *empty* ordered maps as a bare `!!omap` tag. geckopy's reader (via
`raven-python`) needs these written explicitly as `!!omap []`. The small helper
below patches that so the file loads. *(This is a temporary workaround for the
alpha release; report upstream so geckopy's reader tolerates the MATLAB form.)*

In [ ]:
def patch_matlab_yaml(src):
    """Make empty `!!omap` tags explicit so geckopy/raven-python can read a
    GECKO-MATLAB-written ecModel YAML."""
    lines = open(src, encoding='utf-8').read().split('\n')
    out = []
    for i, ln in enumerate(lines):
        if ln.rstrip().endswith(': !!omap'):
            indent = len(ln) - len(ln.lstrip())
            nxt = lines[i + 1] if i + 1 < len(lines) else ''
            is_seq = nxt.lstrip().startswith('- ') and (len(nxt) - len(nxt.lstrip())) > indent
            if not is_seq:
                ln = ln.rstrip() + ' []'   # empty omap -> explicit
        out.append(ln)
    dst = src.replace('.yml', '_patched.yml')
    open(dst, 'w', encoding='utf-8').write('\n'.join(out))
    return os.path.abspath(dst)

ecRhto_yml = patch_matlab_yaml('ecRhto/models/ecRhto.yml')
print('patched ->', ecRhto_yml)

## Build a model adapter

GECKO uses a *model adapter* to store organism-specific parameters. In MATLAB
this is the `ecRhtoAdapter.m` class; in geckopy it is a `ModelAdapter` built
from `ModelParameters`. We set the same values as the MATLAB adapter.

In [ ]:
params = ModelParameters(
    path=os.path.abspath('ecRhto'),
    conv_gem=os.path.abspath('ecRhto/models/rhto.xml'),
    org_name='Rhodotorula toruloides',
    sigma=0.5, p_tot=0.4385, f=0.5, gr_exp=0.18,
    c_source='r_1714',   # glucose exchange
    bio_rxn='r_4041',    # biomass pseudoreaction
    enzyme_comp='cytoplasm',
)
adapter = ModelAdapter(params)
print('adapter for', params.org_name, '| biomass', params.bio_rxn, '| glucose', params.c_source)

## Load the conventional GEM and the ecModel

In [ ]:
gem = load_conventional_gem(adapter)
print(f'Conventional GEM: {len(gem.reactions)} reactions, {len(gem.metabolites)} metabolites, {len(gem.genes)} genes')

ec = load_ec_model(ecRhto_yml, adapter=adapter)
print(f'ecModel:          {len(ec.reactions)} reactions, {len(ec.metabolites)} metabolites, {len(ec.ec.enzymes)} enzymes')

## The protein machinery and its direction

Let's look at the protein-pool exchange and an example enzyme-usage reaction.
Notice the **forward direction and positive bounds** — the opposite of GECKO
MATLAB (see the table at the top).

In [ ]:
ppe = ec.reactions.get_by_id('prot_pool_exchange')
usage = next(r for r in ec.reactions if r.id.startswith('usage_prot_'))
print(f'prot_pool_exchange : {ppe.reaction:28s} bounds={ppe.bounds}')
print(f'{usage.id:19s}: {usage.reaction:28s} bounds={usage.bounds}')
print()
print('The prot_pool_exchange upper bound is the protein budget Ptot*f*sigma (in mg/gDCW):',
      round(ppe.upper_bound, 2))

### Question 10 (cf. MATLAB)

If `r_0001_EXP_1` carried a flux of 20 mmol/gDCW/h, describe the flux through
the associated `usage_prot_*` and `prot_pool_exchange` reactions — and the
sign of each. (Hint: in geckopy these fluxes are **positive**; in MATLAB they
were negative.)

## Stage 3 — model tuning

### Maximum growth rate

Allow unlimited glucose uptake and maximise biomass.

In [ ]:
ec.reactions.get_by_id(params.c_source).lower_bound = -1000   # unconstrained glucose
ec.objective = params.bio_rxn
sol = ec.optimize()
max_growth = sol.fluxes[params.bio_rxn]
print(f'Max growth rate the ecModel can reach: {max_growth:.4f} /h '
      f'(experimental gR_exp = {params.gr_exp} /h)')

The ecModel cannot reach the experimental growth rate: some kcat values are
too low, forcing too much protein usage. Inspect the enzyme usage.

In [ ]:
sol = ec.optimize()
usage = enzyme_usage(ec, sol.fluxes)
report = report_enzyme_usage(ec, usage)
report.top_abs_usage.head(8)

### Sensitivity tuning of kcat values

`sensitivity_tuning` repeatedly increases the most limiting kcat (10-fold by
default) until the model reaches `gr_exp`.

In [ ]:
tuning = sensitivity_tuning(ec)
growth_after = ec.optimize().fluxes[params.bio_rxn]
print(f'Tuned {len(tuning.rxns)} kcat values; growth now {growth_after:.4f} /h')

### Question 15 (cf. MATLAB)

kcat tuning *raises* selected kcats (less protein needed) while sigma-fitting
*lowers* the available protein pool. Describe how these two seem to pull in
opposite directions, and when you would use each.

## Stage 4 — proteomics integration

Load the absolute proteomics data (6 conditions x 2 replicates). The file is in
mmol/gDCW; geckopy expects mg/gDCW, so we convert with the enzyme molecular
weights. We integrate condition **GexpUrea** (exponential growth on glucose) —
the *fifth* condition, index 4 in Python (0-based).

In [ ]:
prot_data = load_prot_data('ecRhto/data/abs_proteomics.txt', repl_per_cond=[2, 2, 2, 2, 2, 2])
print('proteins:', len(prot_data.uniprot_ids), '| conditions:', np.asarray(prot_data.abundances).shape[1])

# convert mmol/gDCW -> mg/gDCW using enzyme MW (g/mol): conc[mmol] * MW[g/mol] = mg
mw_by_id = dict(zip(ec.ec.enzymes, ec.ec.mw))
abund = np.asarray(prot_data.abundances, dtype=float).copy()
for i, pid in enumerate(prot_data.uniprot_ids):
    abund[i, :] *= mw_by_id.get(pid, np.nan)
prot_data.abundances = abund
print('converted proteomics to mg/gDCW')

In [ ]:
CONDITION = 4   # GexpUrea: exponential growth on glucose (5th condition, 0-based index 4)
fill_enz_concs(ec, prot_data, data_col=CONDITION)
constrain_enz_concs(ec)
n = int((~np.isnan(np.asarray(ec.ec.concs, dtype=float))).sum())
print(f'Constrained {n} of {len(ec.ec.enzymes)} enzymes with measured concentrations')

Update the protein pool with a condition-specific f-factor and the measured
total protein content (Ptot) from the flux data, then apply the measured
exchange fluxes (loosely: fluxes may be *below* the measured value).

In [ ]:
f_cond = calculate_f_factor(ec, prot_data)
flux_data = load_flux_data('ecRhto/data/fluxData.tsv')
set_prot_pool_size(ec, p_tot=float(np.asarray(flux_data.p_tot)[CONDITION]), f=f_cond)

apply_flux_data_constraints(ec, flux_data, condition=CONDITION,
                            max_min_growth='max', loose_strict_flux='loose')
sol = ec.optimize()
target = float(np.asarray(flux_data.gr_rate)[CONDITION])
print(f'Growth after proteomics + flux constraints: {sol.objective_value:.4f} /h (experimental {target:.4f})')

If the growth rate is not reached, some measured enzyme concentrations are too
low (membrane proteins are notoriously under-measured). `flexibilize_enz_concs`
raises the most limiting concentrations until the target growth is reached.

In [ ]:
flex = flexibilize_enz_concs(ec, exp_growth=target, fold_change=10.0)
sol = ec.optimize()
print(f'Growth after flexibilizing: {sol.objective_value:.4f} /h '
      f'({len(flex.uniprot_ids)} enzyme concentrations relaxed)')

## Stage 5 — simulation and analysis

### Minimal protein usage

Fix growth at 99% of its maximum and **minimise** the protein pool. Because
`prot_pool_exchange` is a *positive* forward reaction in geckopy, "minimise
protein usage" means minimising that flux — we set its objective coefficient to
`-1` and maximise (equivalently, minimise the flux).

### Question 21 (cf. MATLAB)

In MATLAB you *maximised* `prot_pool_exchange` to minimise the (negative)
protein usage. Why is it the other way around here?

In [ ]:
ec.objective = params.bio_rxn
g = ec.optimize().fluxes[params.bio_rxn]
ec.reactions.get_by_id(params.bio_rxn).lower_bound = 0.99 * g
ec.objective = {ec.reactions.get_by_id('prot_pool_exchange'): -1.0}   # minimise the positive pool flux
sol = ec.optimize()
print(f'Minimum protein pool usage at 99% of max growth: {abs(sol.fluxes["prot_pool_exchange"]):.2f} mg/gDCW')

### Compare ecModel and conventional GEM fluxes

ecModel reactions are split per isoenzyme/direction, so their fluxes can't be
compared one-to-one with the GEM. `map_rxns_to_conv` maps them back to the
conventional reactions.

### Question 22 (cf. MATLAB)

Why can the ecModel and conventional-GEM FBA solution vectors not be compared
directly, reaction by reaction?

In [ ]:
gem.adapter = adapter
sol = ec.optimize()
mapped = map_rxns_to_conv(ec, gem, sol.fluxes)
print('mapped ecModel fluxes back onto', len(gem.reactions), 'conventional reactions:',
      type(mapped).__name__)

### ec-flux variability analysis (optional)

`ec_fva` collapses the isoenzyme/direction splits and reports one (min, max)
per conventional reaction. It is **slow with the default GLPK solver** (minutes
for thousands of reactions x 2 LPs), so it is left commented out — enable a
faster solver (`ec.solver = "gurobi"`) first.

In [ ]:
# from geckopy import ec_fva
# fva = ec_fva(ec, gem)          # returns a DataFrame of min_flux / max_flux per reaction
# fva['range'] = fva['max_flux'] - fva['min_flux']
# fva['range'].sort_values().plot()  # CDF-style variability plot

## Notes on differences from the GECKO MATLAB version

- **Protein-reaction direction (the big one).** geckopy uses the natural
  *forward* direction (positive flux) for `usage_prot_*` and
  `prot_pool_exchange`, the mirror image of MATLAB's reverse/negative
  convention. Minimise (not maximise) the pool to get minimal protein usage;
  apply proteomics as an *upper* bound; expect positive usage fluxes.
- **Reconstruction not shown.** Stages 0-2 (adapter, UniProt/BRENDA/DLKcat,
  kcat assignment) need online databases + Docker. geckopy implements them
  (`make_ec_model`, `fuzzy_kcat_matching`, `run_dlkcat`, …) — see
  `tutorials/full_ecModel/protocol.py` in the geckopy repo — but here we start
  from the pre-built ecModel.
- **YAML compatibility.** A GECKO-MATLAB-written ecModel YAML writes empty
  ordered maps as a bare `!!omap`; geckopy's reader needs `!!omap []`. The
  `patch_matlab_yaml` helper above works around this (alpha-release caveat).
- **Install.** geckopy is alpha and not on PyPI; it is installed from git, and
  currently needs `raven-python@main` rather than the `@develop` branch its
  metadata pins.
- **Solver.** geckopy extends cobrapy and uses GLPK by default; switch with
  `ec.solver = "gurobi"` for speed (FVA in particular).